In [76]:
# Import Libraries
import pandas as pd
import numpy as np
import pyodbc
import warnings
warnings.filterwarnings('ignore')
import copy
import datetime
from functools import reduce

# Import the Data

In [77]:
# Establish the connection 
conn = pyodbc.connect('Driver={ODBC Driver 13 for SQL Server};'
                      'Server=DDAMWSQL16.sandag.org;'
                      'Database=demographic_warehouse;'
                      'Trusted_Connection=yes;')

# Input Data

In [78]:
#2050

In [79]:
year = 2050
staging_table = '[2023_12_05]'

## Sex Table

In [80]:
sex_query = f'''SELECT
      [mgra]
      ,[sex].sex
      ,SUM([hhp]) AS pop
  FROM [sr15_staging].{staging_table}.[pop_ase_mgra]
  LEFT JOIN [demographic_warehouse].[dim].[sex]
  ON [pop_ase_mgra].sex_id = [sex].sex_id
  WHERE increment = {year}
  GROUP BY [mgra], [sex].sex'''

sex_df =  pd.read_sql_query(sex_query, conn)

# make manipulations
sex_df = sex_df.pivot(index='mgra', columns='sex', values='pop').reset_index()
sex_df.columns.name = ''
sex_df = sex_df[['mgra', 'Male', 'Female']]
sex_df

,mgra,Male,Female
0,1,181,198
1,2,10,11
2,3,236,273
3,4,0,0
4,5,50,42
...,...,...,...
24316,24317,0,0
24317,24318,51,71
24318,24319,0,0
24319,24320,0,0


## Age Table

In [81]:
age_query = f'''SELECT 
    mgra,
    CASE 
        WHEN [age_group_id] BETWEEN 1 AND 1 THEN 'Age_LT5'
        WHEN [age_group_id] BETWEEN 2 AND 2 THEN 'Age_5to9'
        WHEN [age_group_id] BETWEEN 3 AND 3 THEN 'Age_10to14'
        WHEN [age_group_id] BETWEEN 4 AND 4 THEN 'Age_15to17'
        WHEN [age_group_id] BETWEEN 5 AND 6 THEN 'Age_18to24'
        WHEN [age_group_id] BETWEEN 7 AND 8 THEN 'Age_25to34'
        WHEN [age_group_id] BETWEEN 9 AND 10 THEN 'Age_35to44'
        WHEN [age_group_id] BETWEEN 11 AND 12 THEN 'Age_45to54'
        WHEN [age_group_id] BETWEEN 13 AND 15 THEN 'Age_55to64'
        WHEN [age_group_id] BETWEEN 16 AND 17 THEN 'Age_65to74'
        WHEN [age_group_id] BETWEEN 18 AND 19 THEN 'Age_75to84'
        WHEN [age_group_id] BETWEEN 20 AND 21 THEN 'Age_85Plus'
        ELSE NULL 
    END AS age_group,
    SUM([hhp]) AS 'pop'
FROM [sr15_staging].{staging_table}.[pop_ase_mgra]
WHERE increment = {year}
GROUP BY [increment], 
    mgra,
    CASE 
        WHEN [age_group_id] BETWEEN 1 AND 1 THEN 'Age_LT5'
        WHEN [age_group_id] BETWEEN 2 AND 2 THEN 'Age_5to9'
        WHEN [age_group_id] BETWEEN 3 AND 3 THEN 'Age_10to14'
        WHEN [age_group_id] BETWEEN 4 AND 4 THEN 'Age_15to17'
        WHEN [age_group_id] BETWEEN 5 AND 6 THEN 'Age_18to24'
        WHEN [age_group_id] BETWEEN 7 AND 8 THEN 'Age_25to34'
        WHEN [age_group_id] BETWEEN 9 AND 10 THEN 'Age_35to44'
        WHEN [age_group_id] BETWEEN 11 AND 12 THEN 'Age_45to54'
        WHEN [age_group_id] BETWEEN 13 AND 15 THEN 'Age_55to64'
        WHEN [age_group_id] BETWEEN 16 AND 17 THEN 'Age_65to74'
        WHEN [age_group_id] BETWEEN 18 AND 19 THEN 'Age_75to84'
        WHEN [age_group_id] BETWEEN 20 AND 21 THEN 'Age_85Plus'
        ELSE NULL
    END;'''

age_df =  pd.read_sql_query(age_query, conn)

# make manipulations
age_df_pivot = age_df.pivot(index='mgra', columns='age_group', values='pop').reset_index()
age_df_pivot.columns.name = ''
age_df_pivot = age_df_pivot[['mgra', 'Age_LT5', 'Age_5to9', 'Age_10to14', 'Age_15to17', 'Age_18to24', 'Age_25to34','Age_35to44', 'Age_45to54', 'Age_55to64', 'Age_65to74','Age_75to84', 'Age_85Plus']]
age_df_pivot

,mgra,Age_LT5,Age_5to9,Age_10to14,Age_15to17,Age_18to24,Age_25to34,Age_35to44,Age_45to54,Age_55to64,Age_65to74,Age_75to84,Age_85Plus
0,1,13,6,21,10,29,19,32,38,53,62,84,12
1,2,0,1,0,0,1,0,4,4,4,4,3,0
2,3,55,42,35,21,27,53,52,64,50,31,44,35
3,4,0,0,0,0,0,0,0,0,0,0,0,0
4,5,2,1,9,6,11,1,14,15,5,13,11,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24316,24317,0,0,0,0,0,0,0,0,0,0,0,0
24317,24318,1,7,14,8,8,10,8,14,11,21,11,9
24318,24319,0,0,0,0,0,0,0,0,0,0,0,0
24319,24320,0,0,0,0,0,0,0,0,0,0,0,0


## Ethnicity 

In [82]:
race_query = f'''SELECT
      [mgra]
      ,[ethnicity].long_name
      ,SUM([hhp]) AS hhp
  FROM [sr15_staging].{staging_table}.[pop_ase_mgra]
  LEFT JOIN [demographic_warehouse].[dim].[ethnicity]
  ON [pop_ase_mgra].ethnicity_id = [ethnicity].ethnicity_id
  WHERE increment = {year}
  GROUP BY mgra, [ethnicity].long_name'''

race_df =  pd.read_sql_query(race_query, conn)
race_df




# # make manipulations
race_df_pivoted = race_df.pivot(index='mgra', columns='long_name', values='hhp').reset_index()
race_df_pivoted.columns.name = ''

race_df_pivoted['Non-Hispanic, Other'] = race_df_pivoted['Non-Hispanic, Other'] + race_df_pivoted['Non-Hispanic, American Indian or Alaska Native'] + race_df_pivoted['Non-Hispanic, Hawaiian or Pacific Islander']
race_df_pivoted = race_df_pivoted.drop(['Non-Hispanic, American Indian or Alaska Native', 'Non-Hispanic, Hawaiian or Pacific Islander'], axis=1)
race_df_pivoted
race_df_pivoted = race_df_pivoted.rename(columns={'Non-Hispanic, Asian':'Asian',
                                'Non-Hispanic, Black':'Black',
                                'Non-Hispanic, Other':'Other_v2',
                                'Non-Hispanic, Two or More Races':'TwoorMore',
                                'Non-Hispanic, White':'White'})

# (Ask Nivedya) This leaves out: 'Non-Hispanic, American Indian or Alaska Native', 'Non-Hispanic, Hawaiian or Pacific Islander', 
race_df_pivoted = race_df_pivoted[['mgra', 'Asian', 'Black', 'Hispanic', 'Other_v2', 'TwoorMore',	'White']]
race_df_pivoted

,mgra,Asian,Black,Hispanic,Other_v2,TwoorMore,White
0,1,152,47,109,1,25,45
1,2,5,0,7,2,1,6
2,3,19,14,136,2,13,325
3,4,0,0,0,0,0,0
4,5,0,0,19,0,1,72
...,...,...,...,...,...,...,...
24316,24317,0,0,0,0,0,0
24317,24318,13,4,19,1,38,47
24318,24319,0,0,0,0,0,0
24319,24320,0,0,0,0,0,0


## HH Size, HH Workers, HH Child

In [83]:
hh_char_query = f'''SELECT
	  mgra
      ,SUM([hhs1]) AS HHSize_1
      ,SUM([hhs2]) AS HHSize_2
	  , SUM([hhs3]) AS HHSize_3
	  ,SUM([hhs4]) + SUM([hhs5]) + SUM([hhs6]) + SUM([hhs7]) AS HHSize_4Plus
	  ,SUM([hhs1]) + SUM([hhs2]) + SUM([hhs3]) + SUM([hhs4]) + SUM([hhs5]) + SUM([hhs6]) + SUM([hhs7]) AS Total_HH
      ,SUM([hhworkers0]) AS HHWork_0
      ,SUM([hhworkers1]) AS HHWork_1
      ,SUM([hhworkers2]) AS HHWork_2
      ,SUM([hhworkers3]) AS HHWork_3Plus
	  ,SUM([hhwoc]) AS HHChild_0
	,SUM([hhwc]) AS HHChild_1Plus
  FROM [sr15_staging].{staging_table}.[hh_characteristics_mgra]
  WHERE increment = {year}
  GROUP BY mgra'''

hh_char_df =  pd.read_sql_query(hh_char_query, conn)
hh_char_df

,mgra,HHSize_1,HHSize_2,HHSize_3,HHSize_4Plus,Total_HH,HHWork_0,HHWork_1,HHWork_2,HHWork_3Plus,HHChild_0,HHChild_1Plus
0,2917,201,190,51,40,482,124,181,135,42,418,64
1,10079,16,21,8,7,52,11,19,15,7,41,11
2,5834,7,14,13,20,54,9,21,17,7,32,22
3,7162,117,103,29,15,264,69,84,83,28,207,57
4,24142,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
24316,11361,0,0,0,0,0,0,0,0,0,0,0
24317,10279,0,0,0,0,0,0,0,0,0,0,0
24318,9197,33,57,33,60,183,51,63,51,18,117,66
24319,10820,2,2,2,5,11,4,4,3,0,6,5


## Income

In [84]:
inc_query = f'''SELECT
      [mgra]
      ,[i1] AS HHInc_0to14999
      ,[i2] AS HHInc_15000to29999
      ,[i3] + i4 AS HHInc_30000to59999
      ,[i5] + [i6] AS HHInc_60000to99999
      ,[i7] + [i8] AS HHInc_100000to149999
      ,[i9] AS HHInc_150000to199999
      ,[i10] AS HHInc_200000Plus
	  ,[gq_civ_college] AS gq_college_pop
	  ,[gq_mil] AS gq_mil_pop
	  ,[gq_civ_other] AS gq_other_pop -- (Ask Nivedya) Is this meant to be GQ civ would that be the other in the mgra_control?
  FROM [sr15_staging].{staging_table}.[mgrabase]
  WHERE increment = {year}'''

inc_df =  pd.read_sql_query(inc_query, conn)
inc_df

,mgra,HHInc_0to14999,HHInc_15000to29999,HHInc_30000to59999,HHInc_60000to99999,HHInc_100000to149999,HHInc_150000to199999,HHInc_200000Plus,gq_college_pop,gq_mil_pop,gq_other_pop
0,4497,4,3,3,9,6,4,4,0,0,0
1,4506,0,0,0,0,0,0,0,0,0,0
2,4531,0,0,0,0,0,0,0,0,0,0
3,4540,0,0,0,0,0,0,0,0,0,0
4,4931,5,4,10,11,15,12,11,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
24316,18091,0,0,0,3,1,3,4,0,0,0
24317,18105,0,0,0,0,2,0,0,0,0,0
24318,18106,0,0,0,3,0,0,0,0,0,0
24319,18141,2,2,6,3,6,0,0,0,0,0


## Add Together

In [85]:
# Put all your DataFrames in a list
dataframes = [sex_df, age_df_pivot, race_df_pivoted, hh_char_df, inc_df]

# Use reduce to merge them all together
output = reduce(lambda left, right: pd.merge(left, right, on='mgra'), dataframes)
output['Total_HH_GQ'] = output['Total_HH'] + output['gq_college_pop'] + output['gq_mil_pop'] + output['gq_other_pop']
output

,mgra,Male,Female,Age_LT5,Age_5to9,Age_10to14,Age_15to17,Age_18to24,Age_25to34,Age_35to44,...,HHInc_15000to29999,HHInc_30000to59999,HHInc_60000to99999,HHInc_100000to149999,HHInc_150000to199999,HHInc_200000Plus,gq_college_pop,gq_mil_pop,gq_other_pop,Total_HH_GQ
0,1,181,198,13,6,21,10,29,19,32,...,17,53,51,18,10,3,0,0,0,167
1,2,10,11,0,1,0,0,1,0,4,...,0,1,2,3,1,4,0,0,54,69
2,3,236,273,55,42,35,21,27,53,52,...,44,53,50,12,9,0,0,0,0,198
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,50,42,2,1,9,6,11,1,14,...,0,4,6,7,6,14,0,0,0,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24316,24317,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24317,24318,51,71,1,7,14,8,8,10,8,...,5,4,8,5,7,14,0,0,0,46
24318,24319,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24319,24320,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [86]:
output.to_csv(rf'outputs/mgra_control_{year}.csv', index=False)